# Road Surface Coverage and Accuracy Analysis

This notebook compares road surface coverage between OSM surface tags and model predictions, and computes accuracy metrics by region and road type.

## 1. Configuration

In [ ]:
from pathlib import Path
import os

# Base directories
BASE_DATA_DIR = Path("/path/to/data")

# Input data
PARQUET_2024 = BASE_DATA_DIR / "road_surfaces" / "2024" / "all_roads_final.parquet"
PARQUET_VALID = BASE_DATA_DIR / "validation" / "DL_tags_validation.parquet"
EXAMPLE_IMAGE = "qualitative_examples_numbered.png"

# Output directory
OUTPUT_DIR = Path("./plots")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Configuration loaded")

## 2. Imports

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from shapely.wkb import loads as from_wkb

## 3. Color Palette

In [ ]:
COLORS = {
    "paved":   "#6A8E5A",
    "unpaved": "#C4A46A",
    "unknown": "#B8B8B8",
}

ACCURACY_COLORS = {
    "osm":   "#5B7FBC",
    "model": "#5AA17E",
}

## 4. Utility Functions

In [ ]:
def normalize_class(x):
    """Normalize surface class to paved/unpaved/unknown."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "unknown"
    x = str(x).strip().lower()
    if x == "paved":
        return "paved"
    if x == "unpaved":
        return "unpaved"
    return "unknown"


def coverage_by_length(df, cls_col, length_col="length"):
    """Compute coverage statistics by road class."""
    total = float(df[length_col].sum())
    classes = ["paved", "unpaved", "unknown"]
    if total <= 0:
        return pd.DataFrame({"class": classes, "pct": [0, 0, 0], "len_m": [0, 0, 0]})
    cov = (
        df.groupby(cls_col, dropna=False)[length_col]
          .sum()
          .reindex(classes)
          .fillna(0.0)
          .rename_axis("class")
          .reset_index(name="len_m")
    )
    cov["pct"] = 100.0 * cov["len_m"] / total
    return cov


def format_scientific(val):
    """Format value in scientific notation."""
    if val == 0 or not np.isfinite(val):
        return "0"
    v = abs(val)
    exp = int(math.floor(math.log10(v)))
    mant = v / (10 ** exp)
    sign = "-" if val < 0 else ""
    return f"{sign}{mant:.2g}×10^{exp}"

## 5. Load Coverage Data

In [ ]:
def load_coverage_2024(parquet_path):
    """Load road data and compute coverage statistics."""
    cols = ["length", "surface_class", "road_class"]
    df = pd.read_parquet(str(parquet_path), columns=cols).copy()
    df["length"] = pd.to_numeric(df["length"], errors="coerce").fillna(0).clip(lower=0).astype(float)
    total_len_m = float(df["length"].sum())
    
    df["osm_cls"] = df["surface_class"].map(normalize_class)
    df["model_cls"] = df["road_class"].map(normalize_class)
    
    cov_osm = coverage_by_length(df, "osm_cls", "length")
    cov_model = coverage_by_length(df, "model_cls", "length")
    
    return cov_osm, cov_model, total_len_m


cov_osm, cov_model, total_len_m = load_coverage_2024(PARQUET_2024)
print(f"Total road length: {total_len_m/1e9:.2f} billion meters")
print(f"\nOSM coverage:\n{cov_osm}")
print(f"\nModel coverage:\n{cov_model}")

## 6. Load Validation Data

In [ ]:
def load_valid_data(parquet_path):
    """Load and process validation dataset."""
    df = pd.read_parquet(str(parquet_path)).copy()
    df.rename(columns=lambda c: c.strip() if isinstance(c, str) else c, inplace=True)
    
    for col in ["road_class", "surface_class", "DL False", "Invalid Data", "Unsure", "validated"]:
        if col not in df.columns:
            df[col] = np.nan
    
    df["DL_OSM_Mismatch"] = (df["road_class"] != df["surface_class"]).astype(int)
    
    def actual_pred(row):
        rc = row["road_class"]
        if pd.isna(row["DL False"]):
            return rc
        if rc == "paved":
            return "unpaved"
        if rc == "unpaved":
            return "paved"
        return rc
    
    df["Actual_pred"] = df.apply(actual_pred, axis=1)
    df["OSM_False"] = (df["surface_class"] != df["Actual_pred"]).astype(int)
    
    valid_mask = df["validated"].fillna(0).astype(int).eq(1)
    if "Invalid Data" in df.columns:
        valid_mask &= df["Invalid Data"].isna()
    if "Unsure" in df.columns:
        valid_mask &= df["Unsure"].isna()
    
    return df.loc[valid_mask].copy()


valid_data = load_valid_data(PARQUET_VALID)
print(f"Loaded {len(valid_data):,} validated records")

## 7. Compute Accuracy by Region

In [ ]:
def compute_region_type_accuracies(valid_data, region_col="continent", type_col="osm_tags_highway",
                                   min_n_cell=15, min_n_region=30):
    """Compute accuracy metrics grouped by region and road type."""
    vd = valid_data.copy()
    
    if type_col in vd.columns:
        vd[type_col] = vd[type_col].astype(str).str.replace(r"_link$", "", regex=True)
    else:
        vd[type_col] = "all"
    
    for col in ["OSM_False", "DL False", "validated", region_col, type_col]:
        if col not in vd.columns:
            vd[col] = np.nan
    
    grp = (
        vd.groupby([region_col, type_col], dropna=False)
          .agg(
              n=("validated", "sum"),
              osm_incorrect=("OSM_False", "sum"),
              dl_incorrect=("DL False", "sum"),
          )
          .reset_index()
    )
    
    grp = grp[grp["n"] >= max(1, min_n_cell)].copy()
    if grp.empty:
        return grp
    
    grp["OSM_Accuracy"] = (1 - grp["osm_incorrect"] / grp["n"]) * 100.0
    grp["Planet_Accuracy"] = (1 - grp["dl_incorrect"] / grp["n"]) * 100.0
    
    reg_sizes = grp.groupby(region_col)["n"].sum()
    keep_regions = reg_sizes[reg_sizes >= min_n_region].index
    grp = grp[grp[region_col].isin(keep_regions)].copy()
    
    return grp


df_acc = compute_region_type_accuracies(valid_data)
print(f"Accuracy data for {df_acc['continent'].nunique()} regions")

## 8. Plotting Functions

In [ ]:
def plot_coverage_stacked(ax, cov_osm, cov_model, colors, title="", total_len_m=None, n_guides=3):
    """Plot stacked bar chart of coverage."""
    classes = ["paved", "unpaved", "unknown"]
    labels = ["OSM", "Planet"]
    guide_color = "#e5e7eb"
    
    osm = cov_osm.set_index("class").reindex(classes)
    mdl = cov_model.set_index("class").reindex(classes)
    osm_len = osm["len_m"].fillna(0).to_numpy(float)
    model_len = mdl["len_m"].fillna(0).to_numpy(float)
    osm_pct = osm["pct"].fillna(0).to_numpy(float)
    model_pct = mdl["pct"].fillna(0).to_numpy(float)
    
    stacks_abs = [osm_len, model_len]
    stacks_pct = [osm_pct, model_pct]
    
    if total_len_m is None or not np.isfinite(total_len_m) or total_len_m <= 0:
        total_len_m = float(max(osm_len.sum(), model_len.sum(), 1.0))
    ymax = total_len_m * 1.05
    ax.set_ylim(0, ymax)
    
    ticks = np.linspace(0, total_len_m, max(1, n_guides) + 2)[1:-1]
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: format_scientific(y)))
    ax.set_ylabel("Total road length (m)")
    ax.set_title(title)
    
    ax.tick_params(axis="y", which="major", length=4, width=0.8, color=guide_color, labelcolor="black")
    ax.grid(False)
    for y in ticks:
        ax.axhline(y, color=guide_color, linestyle=":", linewidth=0.9, zorder=0)
    
    bottoms = np.zeros(2, dtype=float)
    for ci, cls in enumerate(classes):
        vals = np.array([stacks_abs[0][ci], stacks_abs[1][ci]], dtype=float)
        ax.bar(labels, vals, bottom=bottoms, color=colors[cls], edgecolor="none", zorder=1)
        bottoms += vals
    
    for i in range(2):
        bottom = 0.0
        for ci, cls in enumerate(classes):
            v_abs = stacks_abs[i][ci]
            v_pct = stacks_pct[i][ci]
            if v_abs > 0:
                ax.text(i, bottom + v_abs / 2.0, f"{v_pct:.1f}%",
                        ha="center", va="center", fontsize=8,
                        color=("white" if cls == "paved" else "black"), zorder=3)
                bottom += v_abs
    
    for s in ax.spines.values():
        s.set_visible(False)


def plot_accuracy_grid(df_acc, axes, region_col="continent", type_col="osm_tags_highway"):
    """Plot accuracy comparison by region."""
    if df_acc.empty:
        for ax in axes:
            ax.set_visible(False)
        return
    
    type_order = df_acc.groupby(type_col)["n"].sum().sort_values(ascending=False).index.tolist()
    y = np.arange(len(type_order))
    yticks = [str(t) for t in type_order]
    
    regions = df_acc[region_col].astype(str).fillna("Unknown").unique().tolist()
    color_osm = "#2563eb"
    color_mod = "#10b981"
    threshold = 35
    
    for i, region in enumerate(regions):
        if i >= len(axes):
            break
        ax = axes[i]
        sub = df_acc[df_acc[region_col].astype(str) == region].copy()
        sub = sub.set_index(type_col).reindex(type_order)
        
        for col in ["n", "OSM_Accuracy", "Planet_Accuracy"]:
            sub[col] = sub[col].fillna(0)
        
        osm_acc = sub["OSM_Accuracy"].to_numpy()
        dl_acc = sub["Planet_Accuracy"].to_numpy()
        n_vals = sub["n"].to_numpy()
        
        ax.barh(y, -osm_acc, color=color_osm)
        ax.barh(y, dl_acc, color=color_mod)
        
        ax.axvline(0, color="k", linewidth=2)
        ax.set_yticks(y)
        ax.set_yticklabels(yticks)
        ax.tick_params(axis="y", labelsize=8, length=0)
        ax.set_xlim(-100, 110)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
        ax.tick_params(axis="x", length=3)
        for spine in ax.spines.values():
            spine.set_visible(False)
        
        ax.set_title(region.replace("_", " ").title(), fontsize=10)
        
        for yi, (oa, da) in enumerate(zip(osm_acc, dl_acc)):
            if oa > 0:
                ax.text(-oa / 2 if oa >= threshold else -oa - 3, yi, f"{oa:.0f}%",
                        va="center", ha="center" if oa >= threshold else "right",
                        fontsize=7, color="white" if oa >= threshold else "black", clip_on=False)
            if da > 0:
                ax.text(da / 2 if da >= threshold else da + 3, yi, f"{da:.0f}%",
                        va="center", ha="center" if da >= threshold else "left",
                        fontsize=7, color="white" if da >= threshold else "black", clip_on=False)
        
        for yi, n in enumerate(n_vals):
            if n > 0:
                ax.text(105, yi, f"n={int(n)}", va="center", ha="left", fontsize=7, color="black", clip_on=False)
        
        ax.grid(axis="x", linestyle=":", alpha=0.3)
    
    for j in range(len(regions), len(axes)):
        axes[j].set_visible(False)

## 9. Combined Figure

In [2]:
plt.rcParams['font.size'] = '8'

def plot_coverage_and_accuracy(
    valid_data,
    parquet_2024,
    region_col="continent",
    type_col="osm_tags_highway",
    min_n_cell=40,
    min_n_region=30,
    acc_ncols=3,
    savepath=None,
    n_guides=3,
    acc_wspace=0.45,
    acc_hspace=0.36,
    third_image_path=None,
    third_image_title=None,
    fig_width=21.0,
    min_img_h_in=2.5,
    max_img_h_in=10.0,
    group_gap=0.20,
    image_title_pad=12,
):
    """Create combined coverage and accuracy figure."""
    cov_osm, cov_model, total_len_m = load_coverage_2024(parquet_2024)
    df_acc = compute_region_type_accuracies(
        valid_data, region_col=region_col, type_col=type_col,
        min_n_cell=min_n_cell, min_n_region=min_n_region
    )

    n_regions = df_acc[region_col].nunique() if not df_acc.empty else 0
    acc_nrows = max(1, math.ceil(n_regions / acc_ncols)) if n_regions else 1

    row1_h_in = 2.8 + 2.3 * (acc_nrows - 1)

    img = None
    if third_image_path and os.path.exists(third_image_path):
        try:
            img = mpimg.imread(third_image_path)
            ih, iw = img.shape[0], img.shape[1]
            img_aspect = ih / max(iw, 1)
            row2_h_in = fig_width * 0.96 * img_aspect
            row2_h_in = max(min_img_h_in, min(max_img_h_in, row2_h_in)) + 1
            row2_h_in += image_title_pad / 72.0 * 0.6
        except Exception:
            row2_h_in = min_img_h_in
    else:
        row2_h_in = min_img_h_in

    fig_height = row1_h_in + row2_h_in + 0.8

    fig = plt.figure(figsize=(fig_width, fig_height))
    gs = gridspec.GridSpec(
        nrows=2, ncols=2,
        width_ratios=[1.0, acc_ncols],
        height_ratios=[row1_h_in, row2_h_in],
        wspace=float(group_gap), hspace=0.30
    )

    ax_cov = fig.add_subplot(gs[0, 0])
    plot_coverage_stacked(ax_cov, cov_osm, cov_model, COLORS, title="",
                          total_len_m=total_len_m, n_guides=n_guides)

    sub_gs = gridspec.GridSpecFromSubplotSpec(
        nrows=acc_nrows, ncols=acc_ncols, subplot_spec=gs[0, 1],
        wspace=acc_wspace, hspace=acc_hspace
    )
    axes_acc = [fig.add_subplot(sub_gs[r, c]) for r in range(acc_nrows) for c in range(acc_ncols)]
    plot_accuracy_grid(df_acc, axes_acc, region_col=region_col, type_col=type_col)
    for ax in axes_acc:
        ax.tick_params(axis="y", pad=4)

    ax_img = fig.add_subplot(gs[1, :])
    ax_img.set_axis_off()
    if img is not None:
        ih, iw = img.shape[0], img.shape[1]
        ax_img.imshow(img, origin="upper")
        ax_img.set_aspect('equal', adjustable='box')
        ax_img.set_anchor('C')
        ax_img.set_xlim(0, iw)
        ax_img.set_ylim(ih, 0)
    else:
        ax_img.text(0.5, 0.5, "No image provided", ha="center", va="center", fontsize=12)
    if third_image_title:
        ax_img.set_title(f"(c)\n{third_image_title}", pad=image_title_pad, fontsize=10)

    fig.text(0.215, 0.93, "(a)\nCoverage", ha="center", va="top", fontsize=10)
    fig.text(0.665, 0.93, "(b)\nAccuracy by Road Type", ha="center", va="top", fontsize=10)

    plt.tight_layout(rect=[0.02, 0.10, 0.98, 0.95])

    cov_box = ax_cov.get_position(fig)
    acc_boxes = [ax.get_position(fig) for ax in axes_acc if ax.get_visible()]
    if acc_boxes:
        acc_x0 = min(b.x0 for b in acc_boxes)
        y0 = min(cov_box.y0, min(b.y0 for b in acc_boxes))
        y1 = max(cov_box.y1, max(b.y1 for b in acc_boxes))
        x_mid = 0.475 * (cov_box.x1 + acc_x0)
        fig.add_artist(plt.Line2D([x_mid, x_mid], [y0, y1], transform=fig.transFigure,
                                  color="#e5e7eb", linewidth=1.0))

    row1_bottom = min([cov_box.y0] + [b.y0 for b in acc_boxes]) if acc_boxes else cov_box.y0
    image_top = ax_img.get_position(fig).y1
    legend_y = max((row1_bottom + image_top) / 2.0, 0.06)

    cov_center_x = 0.5 * (cov_box.x1 - cov_box.x)
    cov_handles = [
        mpatches.Patch(color=COLORS["paved"], label="Paved"),
        mpatches.Patch(color=COLORS["unpaved"], label="Unpaved"),
        mpatches.Patch(color=COLORS["unknown"], label="Unknown"),
    ]
    fig.legend(handles=cov_handles, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(cov_center_x, legend_y))

    if acc_boxes:
        acc_center_x = 0.5 * (min(b.x0 for b in acc_boxes) + max(b.x1 for b in acc_boxes))
        acc_handles = [
            Line2D([0], [0], color="#2563eb", lw=8, label="OpenStreetMap"),
            Line2D([0], [0], color="#10b981", lw=8, label="Model"),
        ]
        fig.legend(handles=acc_handles, loc="lower center", ncol=2, frameon=False,
                   bbox_to_anchor=(acc_center_x, legend_y))

        line_y = row1_bottom - 0.055
        fig.add_artist(plt.Line2D([0.125, 0.875], [line_y, line_y],
                                  transform=fig.transFigure, color="#e5e7eb", linewidth=1.0))

    fig.suptitle('Comparison of Global Road Surface Types - OpenStreetMap vs. Planet Derived DL predictions', fontsize=12)

    if savepath:
        fig.savefig(savepath, dpi=600, bbox_inches="tight")

    plt.show()

NameError: name 'plt' is not defined

## 10. Generate Figure

In [ ]:
plot_coverage_and_accuracy(
    valid_data=valid_data,
    parquet_2024=str(PARQUET_2024),
    region_col="continent",
    type_col="osm_tags_highway",
    min_n_cell=40,
    min_n_region=30,
    acc_ncols=3,
    n_guides=3,
    acc_wspace=1,
    acc_hspace=0.56,
    third_image_path=str(EXAMPLE_IMAGE),
    third_image_title="Qualitative Examples of Model Predictions",
    fig_width=12,
    group_gap=0.4,
    image_title_pad=12,
    savepath=str(OUTPUT_DIR / "coverage_accuracy_plot.png")
)